# Investigate a fitted candidate: its two halves

A candidate with everything taken from its row of the fits (`data/candidates/candidate_fits.csv`): distance modulus `mu`, isochrone `age` and `z` (Dotter2016 scale), width `sigma_deg` and the endpoints of the fitted track (`endpoint_1_ra`, `endpoint_1_dec`, `endpoint_2_ra`, `endpoint_2_dec`), written by `candidate_fitted_isochrone.ipynb`: in the frame of these endpoints the track is at phi2 = 0. No distance-modulus scan, no width fit: the regions are centred on phi2 = 0, with the fitted width.

Then a sub-population of the candidate, selected twice: the stars with phi1 < 0, then those with phi1 > 0 (`SPLIT_PHI1`). For each half, its on and off regions over that phi1 range only (`cand.define_regions(phi1_range=...)`), its on − off Hess diagram, the photometric errors of its on region against magnitude, and one CSV file (`hess_error_table`) in `figures/outputs/candidates/<id>/split_half` (`cand.split_half_folder`).

Steps:
1. inputs: the fitted values of the candidate;
2. catalog and stars of the window;
3. matched-filter map at the fitted distance modulus;
4. regions of the whole candidate, from the fitted width;
5. the two halves: Hess diagrams, photometric errors, CSV files;
6. summary.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from streameucliddes.analysis.candidate_investigation import CandidateInvestigation, hess_error_table
from streameucliddes.analysis.cmd_match_filter import get_match_filter
from streameucliddes.catalogs.euclid_catalog import default_star_cuts, get_euclid_stars
from streameucliddes.catalogs.stream_candidates import CANDIDATE_FITS_PATH
from streameucliddes.plotting.plot_candidate_investigation import (
    plot_candidate_hess,
    plot_candidate_map,
    plot_phi1_profile,
    plot_photometric_errors,
)

## 1. Inputs

The candidate by its id, and its row of the fits. The endpoints come from the list of candidates (`candidates_info.csv`); the endpoints of the fitted track, the distance modulus, age, Z and sigma from the fits. `from_candidate_id(source="fitted")` takes them, and the sigma in `cand.fitted_sigma_deg` when the fits have both the fitted endpoints and the sigma: the track is then at phi2 = 0, no width fit. Without them (not written yet), an error: run `candidate_fitted_isochrone` first (width fit, then `RECENTRE_ENDPOINTS`). `SPLIT_PHI1`: the phi1 between the two halves (0: the middle of the endpoints). `PHI1_RANGE`: phi1 range of the whole candidate (None: that of the endpoints, as in `candidate_fitted_isochrone`).

In [ ]:
CANDIDATE_ID = 1  # a candidate of candidate_fits.csv with a width (sigma_deg)

FITS_PATH = CANDIDATE_FITS_PATH
ISOCHRONE_MODEL = "Dotter2016"  # the scale of the fitted Z
CATALOG = "euclid_dr1"
STAR_SELECTION = "morphology_cuts"
DUST_CORRECTION = "PLANCK"  # dust correction of the magnitudes, "SFD98" or "PLANCK"
LOAD_MARGIN_DEG = 1.0  # stars read in the (ra, dec) box of the window plus this margin
SPLIT_PHI1 = 0.0  # deg: the halves are phi1 < SPLIT_PHI1 and phi1 > SPLIT_PHI1
PHI1_RANGE = None  # (phi1_min, phi1_max) of the whole candidate, deg; None: the phi1 of the endpoints

fits = pd.read_csv(FITS_PATH).set_index("candidate")
fit_values = fits.loc[CANDIDATE_ID]
print(f"Candidate {CANDIDATE_ID}, fitted values:")
for name in ("mu", "age", "z", "N"):
    print(f"  {name:>3} = {fit_values[name]:.4g}  -{fit_values[name + '_err_lo']:.2g} +{fit_values[name + '_err_hi']:.2g}")

cand = CandidateInvestigation.from_candidate_id(
    CANDIDATE_ID,
    source="fitted",  # endpoints, distance modulus and sigma of the fits
    fits_path=FITS_PATH,
    age=float(fit_values["age"]),
    metallicity=float(fit_values["z"]),
    isochrone_model=ISOCHRONE_MODEL,
    distance_moduli=[float(fit_values["mu"])],  # only the fitted one: no scan
    match_filter=get_match_filter(),
)
if CANDIDATE_ID in [3]:
    cand.padding_phi2_deg = 7
if CANDIDATE_ID == 11:
    cand.padding_phi2_deg = 9
if cand.fitted_sigma_deg is None:
    raise ValueError(f"Candidate {CANDIDATE_ID}: no fitted endpoints and sigma (sigma_deg) in {FITS_PATH}: fit its width "
                     "and write its endpoints (RECENTRE_ENDPOINTS) in candidate_fitted_isochrone.ipynb first.")
print(f"  sigma = {fit_values['sigma_deg']:.3f} ± {fit_values['sigma_deg_err']:.3f} deg, track at phi2 = 0")
print(f"{cand.name}: endpoints {cand.endpoint_1} and {cand.endpoint_2} deg, dm {cand.distance_modulus:.2f}")
cand.isochrone

## 2. Catalog and stars of the window

As in `candidate_fitted_isochrone`: Euclid DR1 south in the (ra, dec) box of the window plus a small margin, then the stars of the window prepared with the fitted isochrone (`cand.prepare`).

In [ ]:
%%time
(ra_min, ra_max), (dec_min, dec_max) = cand.radec_box
ra_range = ((ra_min - LOAD_MARGIN_DEG) % 360.0, (ra_max + LOAD_MARGIN_DEG) % 360.0)  # ra_min > ra_max: across ra = 0
dec_range = (max(dec_min - LOAD_MARGIN_DEG, -90.0), min(dec_max + LOAD_MARGIN_DEG, 90.0))
print(f"Loaded box: ra {ra_range[0]:.1f} to {ra_range[1]:.1f} deg, dec {dec_range[0]:.1f} to {dec_range[1]:.1f} deg")

catalog = get_euclid_stars(CATALOG, cuts=default_star_cuts(CATALOG, STAR_SELECTION), ra_range=ra_range, dec_range=dec_range,
                           dust_correction=DUST_CORRECTION)
cand.prepare(catalog)

## 3. Matched-filter map at the fitted distance modulus

The map on which the regions of the halves are drawn (`cand.compute_zoom_map`).

In [ ]:
if CANDIDATE_ID == 11:
    phi1_range = (-4, 2.5)
elif CANDIDATE_ID == 1:
    phi1_range = (-4, 4.)
else:
    phi1_range = None
phi2_range = (-2, 2) if CANDIDATE_ID in [1, 7] else None

cand.compute_zoom_map(cand.distance_modulus)
fig, ax = plot_candidate_map(cand, show_regions=False, phi1_range=phi1_range, phi2_range=phi2_range)

## 4. Regions of the whole candidate, from the fitted width

On and off regions centred on phi2 = 0, the fitted track in the frame of the fitted endpoints, with the sigma of the fits (`cand.define_regions()`: phi2 = 0 and `cand.fitted_sigma_deg` without a width fit). The phi1 profile of on − off shows the two halves on either side of `SPLIT_PHI1`.

In [ ]:
cand.define_regions(PHI1_RANGE)  # phi2 = 0 and cand.fitted_sigma_deg: no width fit
whole_summary = cand.summary()
whole_phi1_range = cand.regions["phi1_range"]
print(f"Whole candidate: phi1 {whole_phi1_range[0]:.2f} to {whole_phi1_range[1]:.2f} deg, "
      f"areas on {cand.areas['on']:.2f} deg², off {cand.areas['off']:.2f} deg², SNR {whole_summary['snr']:.2f}")
fig, ax = plot_phi1_profile(cand.phi1_profile, cand.regions)
ax.axvline(SPLIT_PHI1, color="0.3", ls=":", label=f"split, $\\phi_1 = {SPLIT_PHI1:g}^\\circ$")
ax.legend(fontsize="small")

## 5. The two halves

The same steps twice, once per half: regions over the phi1 range of the half only (same sigma and centre), on, off and on − off Hess diagrams of every star of the regions (`compute_hess`, as in `candidate_fitted_isochrone`), the fitted isochrone, and the median photometric error of each band of the stars of the on region against its own magnitude (`compute_photometric_errors`). Each half's Hess diagrams and errors go to one CSV file (`hess_error_table`, the columns of `<name>_diff_hess_errors.csv` of `candidate_investigation_example`): `<name>_phi1_negative_diff_hess_errors.csv` and `<name>_phi1_positive_diff_hess_errors.csv` in `cand.split_half_folder`, with the figures of the maps, Hess diagrams (one colour scale for both) and errors.

In [ ]:
HESS_COLOR_BINS = np.arange(-1.0, 1.5 + 0.05, 0.1)  # VIS - H, as candidate_fitted_isochrone
HESS_MAG_BINS = np.arange(18.0, 26.0 + 0.125, 0.25)  # VIS
ERROR_MAG_BINS = np.arange(16.0, 26.25, 0.25)  # from 16: H is brighter

halves = {
    "phi1_negative": {"phi1_range": (whole_phi1_range[0], SPLIT_PHI1), "label": f"$\\phi_1 < {SPLIT_PHI1:g}^\\circ$"},
    "phi1_positive": {"phi1_range": (SPLIT_PHI1, whole_phi1_range[1]), "label": f"$\\phi_1 > {SPLIT_PHI1:g}^\\circ$"},
}
cand.compute_isochrone()  # age, Z and distance modulus of the fits
os.makedirs(cand.split_half_folder, exist_ok=True)  # figures/outputs/candidates/<id>/split_half

fig_maps, map_axes = plt.subplots(2, 1, figsize=(8, 9.5), layout="constrained")
fig_hess, hess_axes = plt.subplots(1, 2, figsize=(11, 5), layout="constrained", sharey=True)
fig_errors, error_axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained", sharex=True, sharey=True)
for (half, info), map_ax, hess_ax, error_ax in zip(halves.items(), map_axes, hess_axes, error_axes):
    cand.define_regions(info["phi1_range"])
    cand.compute_hess(color_bins=HESS_COLOR_BINS, mag_bins=HESS_MAG_BINS)
    cand.compute_photometric_errors(mag_bins=ERROR_MAG_BINS)  # in_match_filter=True: only the match-filter stars
    info["summary"] = cand.summary()
    info["diff_hess"], info["photometric_errors"] = cand.diff_hess, cand.photometric_errors
    snr = info["summary"]["snr"]

    table = hess_error_table(cand.on_hess, cand.off_hess, cand.photometric_errors)
    info["csv_path"] = cand.split_half_folder / f"{cand.name}_{half}_diff_hess_errors.csv"
    table.to_csv(info["csv_path"], index=False)
    print(f"{info['label']}: phi1 {info['phi1_range'][0]:.2f} to {info['phi1_range'][1]:.2f} deg, on {cand.areas['on']:.2f} deg², "
          f"off {cand.areas['off']:.2f} deg², SNR {snr:.2f}; {len(table)} rows written to {info['csv_path']}")

    plot_candidate_map(cand, ax=map_ax, phi1_range=phi1_range, phi2_range=phi2_range)
    map_ax.set_title(f"{map_ax.get_title()}\n{info['label']}")
    plot_candidate_hess(cand, ax=hess_ax, match_filter=True)
    hess_ax.set_title(f"{hess_ax.get_title()}\n{info['label']}")
    hess_ax.text(0.97, 0.03, f"SNR = {snr:.1f}", transform=hess_ax.transAxes, ha="right", va="bottom",
                 bbox={"facecolor": "white", "edgecolor": "0.5", "alpha": 0.8})
    plot_photometric_errors(cand.photometric_errors, ax=error_ax)
    error_ax.set_title(f"{info['label']}: {error_ax.get_title()} of the on region")

# One colour scale for the two on - off diagrams: the larger of the two
vlim = max(max(np.abs(ax.images[0].get_clim())) for ax in hess_axes)
for ax in hess_axes:
    ax.images[0].set_clim(-vlim, vlim)
for fig, kind in ((fig_maps, "onoff"), (fig_hess, "hess"), (fig_errors, "errors")):
    fig.savefig(cand.split_half_folder / f"{cand.name}_split_half_{kind}.png", dpi=150, bbox_inches="tight")

In [ ]:
# The errors of the two halves on the same axes: solid phi1 < 0, dashed phi1 > 0
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
for (half, info), linestyle in zip(halves.items(), ("-", "--")):
    errors = info["photometric_errors"]
    for band, color in zip(errors["bands"], ("C0", "C3")):
        values = np.asarray(errors["error"][band], dtype=float)
        shown = np.isfinite(values)
        ax.plot(np.asarray(errors["mag_centres"])[shown], values[shown], ls=linestyle, marker="o", ms=3, color=color,
                label=f"{band}, {info['label']}")
ax.set_xlabel("magnitude [mag]")
ax.set_ylabel(f"{errors['statistic']} error [mag]")
ax.set_title(f"{cand.name.replace('_', ' ')}: photometric errors of the on region")
ax.legend(fontsize="small")
fig.savefig(cand.split_half_folder / f"{cand.name}_split_half_errors_compared.png", dpi=150)

## 6. Summary

The whole candidate and its two halves. `cand` keeps the regions of the last half (phi1 > 0): `cand.define_regions(PHI1_RANGE)` for those of the whole candidate again.

In [ ]:
columns = ["phi1_range", "area_on_deg2", "area_off_deg2", "n_on", "n_off", "n_excess", "n_excess_error", "snr"]
summary_table = pd.DataFrame({"whole": whole_summary, **{half: info["summary"] for half, info in halves.items()}}).loc[columns]
summary_table